# ZUBAIR LLM — PHASE 1: COMPREHENSIVE AUDIT & BACKUP
**Google Colab GPU Runtime (T4 / A100)**

This notebook performs a **100% non-destructive, read-only** audit and backup:
1. Verifies Google Colab GPU & environment.
2. Mounts Google Drive (`/content/drive/MyDrive/ZUBAIR_LLM`).
3. **Automatic Backup**: Backs up all existing notebooks (`Untitled1.ipynb`, etc.) and Python scripts to `backups/phase1_original/`.
4. **Checkpoint Manifest**: Inspects `best.pt`, `latest.pt`, etc. (shapes, parameter count ~9.1M, epoch, state dict keys) without modifying anything.
5. **Tokenizer Audit**: Checks vocab size, special tokens (`PAD=0, UNK=1, BOS=2, EOS=3, USER=4, ASSISTANT=5`).
6. **Dataset Audit**: Validates line counts, JSON formatting, and empty fields.
7. **Extracts Existing Pipeline**: Dumps the exact `make_batch`, model definition, and generation functions for analysis.

**How to run:** In Colab menu, just click **Runtime -> Run all**.

### Step 1: Environment & GPU Verification

In [ ]:
import sys
import torch

print("=" * 70)
print("STEP 1: HARDWARE & ENVIRONMENT CHECK")
print("=" * 70)
print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
cuda_ok = torch.cuda.is_available()
print(f"CUDA Available : {cuda_ok}")
if cuda_ok:
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory     : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("\n[!] ERROR: No GPU assigned. Please go to Runtime -> Change runtime type -> T4 GPU.")

### Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/ZUBAIR_LLM')
COLAB_NOTEBOOKS_DIR = Path('/content/drive/MyDrive/Colab Notebooks')

print("\n" + "=" * 70)
print("STEP 2: GOOGLE DRIVE VERIFICATION")
print("=" * 70)
print(f"Project Root: {PROJECT_ROOT} | Exists: {PROJECT_ROOT.exists()}")
print(f"Colab Dir   : {COLAB_NOTEBOOKS_DIR} | Exists: {COLAB_NOTEBOOKS_DIR.exists()}")

### Step 3: Run Full Audit, Backup & Extraction

In [ ]:
import os
import json
import hashlib

def sha256_file(p, max_bytes=50*1024*1024):
    h = hashlib.sha256()
    try:
        with open(p, 'rb') as f:
            bytes_read = 0
            while chunk := f.read(1024 * 1024):
                h.update(chunk)
                bytes_read += len(chunk)
                if bytes_read >= max_bytes:
                    break
        return h.hexdigest()
    except Exception as e:
        return f"Error: {e}"

# 1. AUTOMATIC BACKUP OF SCRIPTS & NOTEBOOKS
backup_dir = PROJECT_ROOT / "backups" / "phase1_original"
backup_dir.mkdir(parents=True, exist_ok=True)
backed_up_files = []

# Search and backup existing notebooks
search_dirs = [COLAB_NOTEBOOKS_DIR, PROJECT_ROOT]
for sdir in search_dirs:
    if sdir.exists():
        for f in sdir.glob("*.ipynb"):
            dest = backup_dir / f.name
            shutil.copy2(f, dest)
            backed_up_files.append(f"{f.name} (from {sdir.name})")

print("=" * 70)
print("STEP 3A: AUTOMATIC BACKUP COMPLETE")
print(f"Backup saved safely in: {backup_dir}")
print(f"Files backed up: {backed_up_files}")
print("=" * 70)

# 2. CHECKPOINTS AUDIT
print("\n" + "=" * 70)
print("STEP 3B: CHECKPOINTS AUDIT")
print("=" * 70)
ckpt_candidates = [
    "checkpoints/stage3_scratch_v1/best.pt",
    "checkpoints/stage3_scratch_resume_v1/best.pt",
    "checkpoints/stage3_quality_finetune_v2/best.pt"
]
for f in PROJECT_ROOT.glob("checkpoints/**/*.pt"):
    rel = str(f.relative_to(PROJECT_ROOT)).replace("\\", "/")
    if rel not in ckpt_candidates:
        ckpt_candidates.append(rel)

checkpoints_summary = {}
for rel in ckpt_candidates:
    cp = PROJECT_ROOT / rel
    if not cp.exists():
        print(f"  [-] {rel}: NOT FOUND")
        continue
    sz_mb = round(cp.stat().st_size / (1024 * 1024), 2)
    try:
        data = torch.load(str(cp), map_location='cpu')
        sd = data.get("model_state_dict", data.get("state_dict", data.get("model", data)))
        params = sum(v.numel() for v in sd.values() if hasattr(v, "numel")) if isinstance(sd, dict) else 0
        epoch = data.get("epoch", data.get("current_epoch", "N/A"))
        keys = list(data.keys()) if isinstance(data, dict) else ["raw_weights"]
        checkpoints_summary[rel] = {
            "size_mb": sz_mb,
            "params": params,
            "epoch": epoch,
            "keys": keys
        }
        print(f"  [+] {rel}")
        print(f"      Size: {sz_mb} MB | Params: {params:,} | Epoch: {epoch} | Keys: {keys}")
    except Exception as e:
        print(f"  [!] {rel} load error: {e}")

# 3. TOKENIZER AUDIT
print("\n" + "=" * 70)
print("STEP 3C: TOKENIZER AUDIT")
print("=" * 70)
tok_path = PROJECT_ROOT / "tokenizer/zubair_bpe_v3.json"
tokenizer_summary = {}
if tok_path.exists():
    try:
        with open(tok_path, "r", encoding="utf-8") as f:
            tdata = json.load(f)
        vocab = tdata.get("model", {}).get("vocab", {})
        added = tdata.get("added_tokens", [])
        tokenizer_summary = {
            "vocab_size": len(vocab),
            "added_tokens_count": len(added),
            "added_tokens": added
        }
        print(f"  [+] Tokenizer: {tok_path.name}")
        print(f"      Vocab Size: {len(vocab)} | Added Tokens: {len(added)}")
        for sp in added[:6]:
            print(f"      Special Token: id={sp.get('id')} content='{sp.get('content')}'")
    except Exception as e:
        print(f"  [!] Error reading tokenizer: {e}")
else:
    print(f"  [-] Tokenizer not found at {tok_path}")

# 4. DATASETS AUDIT
print("\n" + "=" * 70)
print("STEP 3D: DATASETS AUDIT")
print("=" * 70)
target_ds = [
    "dataset/cleaned/expanded_v1.jsonl",
    "dataset/splits_expanded_v1/train.jsonl",
    "dataset/splits_expanded_v1/validation.jsonl",
    "dataset/splits_expanded_v1/test.jsonl",
    "dataset/cleaned/targeted_quality_v2.jsonl",
    "dataset/targeted_v2_splits/train.jsonl",
    "dataset/targeted_v2_splits/validation.jsonl"
]
dataset_summary = {}
for ds_rel in target_ds:
    dp = PROJECT_ROOT / ds_rel
    if not dp.exists():
        print(f"  [-] {ds_rel}: NOT FOUND")
        continue
    valid_c, invalid_c = 0, 0
    with open(dp, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    json.loads(line)
                    valid_c += 1
                except:
                    invalid_c += 1
    sz_mb = round(dp.stat().st_size / (1024 * 1024), 2)
    dataset_summary[ds_rel] = {"valid": valid_c, "invalid": invalid_c, "size_mb": sz_mb}
    print(f"  [+] {ds_rel}: {valid_c:,} lines (Size: {sz_mb} MB, Invalid: {invalid_c})")

# 5. SAVE AUDIT REPORT TO GOOGLE DRIVE
audit_report = {
    "backed_up_files": backed_up_files,
    "checkpoints": checkpoints_summary,
    "tokenizer": tokenizer_summary,
    "datasets": dataset_summary
}
report_path = PROJECT_ROOT / "experiments" / "audit_phase1_inventory_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2)

print("\n" + "=" * 70)
print("ALL CHECKS & BACKUPS FINISHED SUCCESSFULLY!")
print(f"Audit report saved to: {report_path}")
print("=" * 70)